# 02 — Graph Workflows: Building a Resume Screening Pipeline

In this notebook we move beyond simple sequential workflows and build a **directed graph** using the `GraphBuilder` DSL.

## What is a Graph Workflow?

A graph workflow is a directed acyclic (or cyclic) graph where:
- **Nodes** are agent invocations with their own params, retry, timeout, and reflection settings
- **Edges** connect nodes and can carry optional conditional expressions
- The orchestrator traverses the graph starting at the **entry node**, following edges whose conditions evaluate to `True`

## Pipeline Architecture

```
parse_resume → match_skills → aggregate_scores → compile_report
 (ResumeParserAgent) (SkillMatcherAgent)  (ScoreAggregatorAgent) (ReportCompilerAgent)
```

This 4-node linear graph represents a resume screening pipeline.

In [ ]:
import time
import json
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)

assert client.ping(), "Orchestrator not reachable — start the server first."
print("Connected to orchestrator.")

## 1. Build the Graph Definition

The `GraphBuilder` uses a fluent API:
1. `.node(id)` — starts a node builder (returns a `GraphNodeBuilder`)
2. `.agent(name)` — binds an agent to the node
3. `.params(**kw)` — passes static parameters to the agent
4. `.timeout(sec)` — per-node timeout in seconds
5. `.done()` — finalises the node and returns to the `GraphBuilder`
6. `.edge(src, tgt)` — adds a directed edge
7. `.entry(id)` — sets the entry-point node
8. `.build()` — produces the raw graph definition dict

In [ ]:
graph_def = (
    GraphBuilder()
    # Node 1: Parse the raw resume text into structured fields
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json", extract_skills=True)
        .timeout(30)
        .done()
    # Node 2: Match extracted skills against a job description
    .node("match_skills")
        .agent("SkillMatcherAgent")
        .params(job_title="Senior Python Engineer", min_match_score=0.6)
        .timeout(30)
        .done()
    # Node 3: Aggregate scores from previous nodes into a single ranking
    .node("aggregate_scores")
        .agent("ScoreAggregatorAgent")
        .params(weights={"skills": 0.5, "experience": 0.3, "education": 0.2})
        .timeout(20)
        .done()
    # Node 4: Compile a human-readable report
    .node("compile_report")
        .agent("ReportCompilerAgent")
        .params(format="markdown", include_recommendations=True)
        .timeout(30)
        .done()
    # Edges — linear pipeline
    .edge("parse_resume",    "match_skills")
    .edge("match_skills",    "aggregate_scores")
    .edge("aggregate_scores", "compile_report")
    # Configuration
    .entry("parse_resume")
    .max_cycles(10)
    .build()
)

print("Graph definition built — nodes:", [n["id"] for n in graph_def["nodes"]])
print("\nFull definition:")
print(json.dumps(graph_def, indent=2))

## 2. Submit the Graph Workflow

`run_graph()` posts the graph definition to `POST /graph/run` and returns an `instance_id`.

The payload carries the input data that agents can read from context.

In [ ]:
# Sample resume payload — in production this would be the actual resume text/data
payload = {
    "resume_text": (
        "Jane Smith\n"
        "5 years Python experience, FastAPI, PostgreSQL, Docker, Kubernetes.\n"
        "BSc Computer Science, MIT, 2018."
    ),
    "candidate_id": "candidate-001",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Graph workflow launched — instance_id: {instance_id}")

## 3. Poll Node States During Execution

`get_node_state(workflow_id, node_id)` retrieves the persisted output for a specific node. We iterate through all 4 nodes and display their state as they complete.

In [ ]:
nodes_in_order = ["parse_resume", "match_skills", "aggregate_scores", "compile_report"]

def poll_node(client, instance_id, node_id, max_wait=30):
    """Wait until a node has output, then return its NodeState."""
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            ns = client.get_node_state(instance_id, node_id)
            if ns.output:
                return ns
        except Exception:
            pass  # node not yet written
        time.sleep(1)
    return None

print(f"Polling node states for workflow {instance_id}:\n")
node_results = {}

for node_id in nodes_in_order:
    print(f"  Waiting for node '{node_id}' ...", end=" ")
    ns = poll_node(client, instance_id, node_id)
    if ns:
        node_results[node_id] = ns.output
        print(f"DONE (updated_at={ns.updated_at})")
    else:
        print("TIMEOUT — node did not complete within 30s")

print("\nAll nodes polled.")

## 4. Display Per-Node Outputs

Each node's output is stored as a dictionary in MongoDB. Here we pretty-print what each agent returned.

In [ ]:
print("=" * 60)
for node_id, output in node_results.items():
    print(f"\nNode: {node_id}")
    print("-" * 40)
    pprint.pprint(output, indent=2)
print("=" * 60)

## 5. Retrieve the Complete Final Graph State

`get_state()` returns all nodes and their outputs in one call — useful for a final snapshot.

In [ ]:
final_state = client.get_state(instance_id)

print(f"Final workflow state")
print(f"  workflow_id  : {final_state.workflow_id}")
print(f"  nodes_stored : {final_state.count}")
print(f"  nodes        : {[n.node_id for n in final_state.nodes]}")

## 6. View Execution Metrics

In [ ]:
metrics = client.get_metrics(instance_id)
print("Graph Execution Metrics")
print("-" * 30)
print(f"  nodes_executed       : {metrics.nodes_executed}")
print(f"  nodes_skipped        : {metrics.nodes_skipped}")
print(f"  reflections_triggered: {metrics.reflections_triggered}")
print(f"  fan_outs_executed    : {metrics.fan_outs_executed}")
print(f"  circuit_breaker_trips: {metrics.circuit_breaker_trips}")
print(f"  error_count          : {metrics.error_count}")

In [ ]:
client.close()
print("Client closed.")

---

## Summary

| Concept | API |
|---------|-----|
| Build graph | `GraphBuilder().node(...).agent(...).params(...).timeout(...).done().edge(...).entry(...).build()` |
| Submit | `client.run_graph(graph_def=..., payload=...)` |
| Per-node output | `client.get_node_state(instance_id, node_id)` |
| Full snapshot | `client.get_state(instance_id)` |
| Metrics | `client.get_metrics(instance_id)` |

**Next**: See `03_signals_control.ipynb` to learn how to interrupt, inject, jump to, and skip nodes at runtime.